In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# --- 1. SETUP & LOAD ---
# Connect to the src folder
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import config

# Load the Ed Sheeran file we created earlier
input_path = config.PROJ_ROOT / "data" / "raw" / "ed_sheeran_charts.csv"
print(f"📂 Loading data from: {input_path}")
df = pd.read_csv(input_path)

# Convert Date column to datetime objects immediately
df['date'] = pd.to_datetime(df['date'])


Config loaded. Pointing to raw data at: /Users/joelangstaff/Downloads
📂 Loading data from: /Users/joelangstaff/Library/Mobile Documents/com~apple~CloudDocs/Code/machine-learning/TOPICS/Time-Series Forecasting/Projects/spotify-forecasting/data/raw/ed_sheeran_charts.csv


In [ ]:

# --- 2. THE INSPECTION (Your Request) ---
print("\n" + "="*40)
print("DATA INSPECTION REPORT")
print("="*40)

print(f"\n1. SHAPE (Rows, Columns): {df.shape}")

print("\n2. INFO (Data Types & Missing Values):")
print("-" * 30)
df.info()

print("\n3. HEAD (First 5 Rows):")
print("-" * 30)
display(df.head())

print("\n4. DESCRIBE (Statistical Summary):")
print("-" * 30)
# formatting floats to 2 decimal places for readability
display(df.describe().round(2))

print("\n" + "="*40)
print("CATEGORICAL DATA REPORT")
print("="*40)

# 1. Unique Count and Values for key columns
for col in ['region', 'chart']: # Added 'chart' to see the split
    if col in df.columns:
        print(f"\nUnique values in '{col}' ({df[col].nunique()} total):")
        print(df[col].unique())


In [ ]:
# --- 3. THE "HIGH CONTRAST" PLOT ---
# We filter for his biggest song to make the chart meaningful
target_song = "Shape of You"
target_region = "Global"

# Sort by date to ensure the line connects properly
subset = df[
    (df['title'] == target_song) &
    (df['region'] == target_region)
].sort_values('date')

# Create the Rolling Average (The Trend)
subset['7_day_avg'] = subset['streams'].rolling(7).mean()

# Plotting
plt.figure(figsize=(14, 7))

# Layer 1: Raw Data (Faded Blue)
sns.lineplot(
    data=subset,
    x='date',
    y='streams',
    label='Daily Streams (Raw)',
    color='steelblue',
    alpha=0.25,       # Transparency makes it sit in the background
    linewidth=1
)

# Layer 2: Trend Line (Bright Orange)
sns.lineplot(
    data=subset,
    x='date',
    y='7_day_avg',
    label='7-Day Trend',
    color='#FF4500',  # Bright Orange-Red
    linewidth=2.5     # Thicker line to pop out
)

plt.title(f"Stream History: {target_song} ({target_region})", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Streams (Millions)", fontsize=12)
plt.grid(True, alpha=0.2)
plt.legend(fontsize=12)
plt.show()


The graph shpws a decay curve as expected. The trend is a raod rise followed by a slow, smooth decline. This already looks sutiable for XGBoost

**I found a critical anomaly in the "Health Check":**

"WARNING: Missing -50 days of data."

Meaning I have duplicate data, which a time series cannot have. This must be fixed this before I can build features, or the model will crash.

In the feature engineering phase, I need to do two things:

1) Remove the duplicates so we have exactly 1 row per day.

2) Create "Lag Features".
- To predict "Streams Tomorrow," the most powerful clue is "Streams Today" and "Streams Last Week." We will create new columns for these.